# 03 - Integração com LLM (Gemini)

Este notebook demonstra a integração do **Integrante 2**: geração de explicações de
diagnóstico em linguagem natural (`src/llm`), usando o modelo treinado pelo
Integrante 3 (Fase 1) e os resultados do algoritmo genético do Integrante 1 (Fase 2).

Cobre os dois casos de uso pedidos no Tech Challenge:
1. Explicação individual de uma predição de risco de AVC, para um paciente específico.
2. Interpretação agregada dos resultados do AG (baseline vs. otimizado), em insights
   acionáveis para profissionais de saúde.

Cada resposta gerada passa por um checklist determinístico de qualidade
(`evaluate_explanation`), sem depender de uma segunda chamada de LLM.

**Pré-requisitos:**
- Rodar `notebooks/01_baseline.ipynb` primeiro (gera os `.joblib` em `results/`).
- Rodar `notebooks/02_genetic_algorithm.ipynb` (gera `results/ga_summary.json`).
- Ter uma `GOOGLE_API_KEY` válida em um arquivo `.env` na raiz do projeto
  (copie `.env.example` e preencha).

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

# Ensure src/ is on the path when running from notebooks/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.llm import evaluate_explanation, explain_prediction, summarize_experiment
from src.models import load_model, predict
from src.preprocessing import prepare_pipeline

ROOT = Path('..').resolve()
ga_summary_path = ROOT / 'results' / 'ga_summary.json'

if not ga_summary_path.exists():
    raise FileNotFoundError(
        'ga_summary.json nao encontrado. Rode notebooks/02_genetic_algorithm.ipynb primeiro.'
    )

with open(ga_summary_path, 'r', encoding='utf-8') as f:
    ga_summary = json.load(f)

print(f"Experimentos carregados: {len(ga_summary['experiments'])}")

## 1. Explicações individuais de diagnóstico

Usamos o modelo de Regressão Logística (maior recall entre os baselines da Fase 1)
para prever o risco de 3 pacientes do conjunto de teste: os 2 com maior probabilidade
estimada de AVC, e 1 com a menor, para contraste.

In [ ]:
model = load_model('logistic_regression')
_, X_test, _, y_test = prepare_pipeline()

predictions, probabilities = predict(model, X_test)
prob_series = pd.Series(probabilities, index=X_test.index)

sample_idx = list(prob_series.sort_values(ascending=False).index[:2]) + \
    list(prob_series.sort_values(ascending=True).index[:1])

In [ ]:
results_individual = []

for idx in sample_idx:
    row_pos = X_test.index.get_loc(idx)
    patient_data = X_test.loc[idx].to_dict()
    prediction = int(predictions[row_pos])
    probability = float(probabilities[row_pos])

    explanation = explain_prediction(patient_data, prediction, probability)
    evaluation = evaluate_explanation(explanation, patient_data)

    results_individual.append({
        'patient_index': idx,
        'prediction': prediction,
        'probability': probability,
        'explanation': explanation,
        'quality_score': evaluation['score'],
        'quality_checks': evaluation['checks'],
    })

    print(f"--- Paciente {idx} (predicao={prediction}, prob={probability:.1%}) ---")
    print(explanation)
    print(f"Qualidade: {evaluation['score']:.2f} | {evaluation['checks']}\n")

## 2. Interpretação agregada dos experimentos do AG

Para cada um dos 3 experimentos registrados pelo Integrante 1 em
`results/ga_summary.json`, comparamos o modelo baseline com o modelo otimizado
pelo algoritmo genético e pedimos um resumo executivo em linguagem natural.

In [ ]:
results_aggregate = []

for exp in ga_summary['experiments']:
    baseline_metrics = ga_summary['baseline'][exp['model_type']]
    optimized_metrics = exp['optimized_metrics']
    best_params = exp['ga']['best_params']

    summary_text = summarize_experiment(baseline_metrics, optimized_metrics, best_params)

    source_data = {**baseline_metrics, **optimized_metrics, **best_params}
    evaluation = evaluate_explanation(summary_text, source_data)

    results_aggregate.append({
        'experiment': exp['name'],
        'model_type': exp['model_type'],
        'summary': summary_text,
        'quality_score': evaluation['score'],
        'quality_checks': evaluation['checks'],
    })

    print(f"--- {exp['name']} ({exp['model_type'].upper()}) ---")
    print(summary_text)
    print(f"Qualidade: {evaluation['score']:.2f} | {evaluation['checks']}\n")

## 3. Resumo da avaliação de qualidade

Consolida o score do checklist determinístico (`src/llm/evaluation.py`) para todas
as respostas geradas nesta demonstração, individuais e agregadas.

In [ ]:
quality_rows = (
    [
        {'tipo': 'individual', 'id': str(r['patient_index']), **r['quality_checks'], 'score': r['quality_score']}
        for r in results_individual
    ]
    + [
        {'tipo': 'agregado', 'id': r['experiment'], **r['quality_checks'], 'score': r['quality_score']}
        for r in results_aggregate
    ]
)

df_quality = pd.DataFrame(quality_rows)
df_quality

## 4. Notas de prompt engineering e avaliação

- **Grounding**: os prompts (`src/llm/prompts.py`) só incluem dados que já existem no
  dataset ou nos resultados do AG — nenhum contexto médico externo é injetado, para
  reduzir o risco de alucinação.
- **Instrução explícita de cautela clínica**: os prompts pedem explicitamente que o
  modelo não substitua o julgamento clínico e não invente exames/históricos.
- **Avaliação determinística vs. LLM-as-judge**: optamos por um checklist de regras
  (`evaluate_explanation`) em vez de usar uma segunda chamada de LLM como "juiz". Isso
  torna a avaliação 100% reprodutível e testável em CI, ao custo de ser mais rígida —
  ela não captura nuances de qualidade que fogem das 4 regras verificadas
  (menção ao recall quando aplicável, grounding numérico, linguagem de risco e
  tamanho da resposta).
- **Limitação conhecida**: o checklist é heurístico. Uma resposta pode ter score alto
  e ainda assim conter uma explicação clinicamente pobre (ou vice-versa) — ele serve
  como um filtro automático de sanidade, não como substituto de revisão humana.